# PAD-UFES-20 + HAM10000 + SCIN 파인튜닝 (8 클래스)

정리된 노트북입니다. 위에서 아래로 순서대로 실행하세요.

기존 7개(색소성 병변)에 SCIN 의 **염증성 질환**을 한 클래스로 더해 8개로 넓힙니다.
지금 모델은 습진 사진을 넣어도 색소성 병변 7개 중 하나로 억지 분류합니다 —
그 빈칸을 메우는 것이 이번 학습의 목적입니다.

**왜 4개가 아니라 1개인가**
직전 실험(11클래스)에서 습진/접촉피부염/두드러기/벌레물림을 서로 구분하는 것은
재현율 0.26~0.45 로 실패했습니다. 반면 "색소성이냐 염증성이냐"는 98.7% 로 맞혔습니다.
SCIN 라벨 자체가 피부과 의사 3인의 **가중 투표**라 이 넷의 경계는 데이터 안에서부터
흐립니다. 모델이 실제로 할 수 있는 구분만 시킵니다.

**실행 전에 준비할 것**
- ⚠️ **`fastapi/tests/baselines/holdout.csv` 를 Drive `MyDrive/artifact_medical_ai/` 폴더에 업로드** — 이걸 빠뜨리면 6번 셀에서 멈춥니다.
  이 목록의 사진은 학습에서 제외되고, 나중에 이 목록으로만 점수를 잽니다.
  안 올리고 학습하면 지난 모델과 비교가 불가능해집니다.
- Drive `MyDrive/artifact_medical_ai/` 폴더에 `pad-ufes-20-small.zip` 업로드
- 같은 폴더에 `scin-small.zip` 업로드 (SCIN 1,972장)
- 같은 폴더에 (선택) 기존 서비스 `model.pth` 업로드
  — 클래스 수가 7→8 로 바뀌어 **분류 헤드는 새로 만들고 백본만 물려받습니다**(셀 9)
- Kaggle API 토큰 준비 (kaggle.com/settings → API → Create New Token)
- 런타임 → 런타임 유형 변경 → **T4 GPU**

In [ ]:
# 1. 한글 폰트 설치 (그래프에 한글 깨짐 방지)
!apt-get install -y fonts-nanum -q
!fc-cache -fv -q

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정 완료")

In [ ]:
# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/artifact_medical_ai'

In [ ]:
# 3. PAD-UFES-20 / SCIN 압축 해제
# 반드시 /content 에 푼다. Drive 에 풀면 학습이 10배 느려진다.
!mkdir -p /content/data
!unzip -q $DRIVE/pad-ufes-20-small.zip -d /content/data
!unzip -q $DRIVE/scin-small.zip -d /content/data

PAD_DIR  = '/content/data/pad-ufes-20-small'
SCIN_DIR = '/content/data/scin-small'
!ls $PAD_DIR/images  | wc -l   # 2298 이 나와야 한다
!ls $SCIN_DIR/images | wc -l   # 1972 가 나와야 한다

In [ ]:
# 4. HAM10000 다운로드 (Kaggle)
# kaggle.com/settings -> API -> Create New Token 으로 받은 토큰을 붙여넣는다.
# 셀에 직접 코드로 적지 않는다 -- 노트북이 Drive 에 저장될 때 평문으로 같이 저장된다.
import os
from getpass import getpass

os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token (KGAT_...): ')

!pip -q install -U kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content

!mkdir -p /content/data/ham10000
!unzip -q /content/skin-cancer-mnist-ham10000.zip -d /content/data/ham10000

# 이미지가 part_1 / part_2 두 폴더로 나뉘어 있어 하나로 합친다
!mkdir -p /content/data/ham10000/images
!mv /content/data/ham10000/HAM10000_images_part_*/* /content/data/ham10000/images/

HAM_DIR     = '/content/data/ham10000'
HAM_IMG_DIR = f'{HAM_DIR}/images'
!ls $HAM_IMG_DIR | wc -l   # 10015 가 나와야 한다

In [ ]:
# 5. 세 데이터셋 병합
import pandas as pd

# fastapi/main.py 의 CLASSES 와 순서까지 정확히 같아야 한다.
# 순서가 어긋나면 배포 후 병명이 통째로 뒤바뀐다.
#
# 앞의 7개는 기존 배포 모델과 인덱스가 같아야 하므로 절대 건드리지 않고,
# 새 클래스는 반드시 뒤에만 덧붙인다.
CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc',   # 색소성 (기존)
           'inflammatory']                                      # 염증성 (신규, 통합)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}

# -- HAM10000 (더모스코피) --
ham = pd.read_csv(f'{HAM_DIR}/HAM10000_metadata.csv')
ham_df = pd.DataFrame({
    'path' : ham['image_id'].apply(lambda x: f'{HAM_IMG_DIR}/{x}.jpg'),
    'label': ham['dx'],
    'group': 'ham_' + ham['lesion_id'].astype(str),   # 같은 병변 묶기
    'src'  : 'ham',
})

# -- PAD-UFES-20 (스마트폰) --
# SCC(편평세포암)와 ACK(광선각화증)는 둘 다 akiec 로 간다.
# HAM10000 의 akiec 정의가 '광선각화증 + 상피내암'이라 원래 둘 다 포함이다.
PAD2HAM = {'BCC': 'bcc', 'ACK': 'akiec', 'SCC': 'akiec',
           'NEV': 'nv',  'SEK': 'bkl',   'MEL': 'mel'}

pad = pd.read_csv(f'{PAD_DIR}/metadata.csv')
pad_df = pd.DataFrame({
    'path' : pad['img_id'].apply(lambda x: f'{PAD_DIR}/images/{x}'),
    'label': pad['diagnostic'].map(PAD2HAM),
    'group': 'pad_' + pad['lesion_id'].astype(str),
    'src'  : 'pad',
})

# -- SCIN (일반인 스마트폰 제출) --
# metadata.csv 에는 4개 라벨이 그대로 들어 있지만, 전부 inflammatory 하나로 합친다.
# 근거: 11클래스 실험에서 이 넷을 서로 구분하는 재현율은 0.26~0.45 였던 반면
#       염증성/색소성 판별은 98.7% 였다. 못 하는 구분을 시키면 mel 까지 같이 망가진다.
# 원본 라벨은 scin_label 로 남겨 둔다 -- 나중에 세분화를 재시도할 때 필요하다.
SCIN2CLS = {'eczema': 'inflammatory', 'acd': 'inflammatory',
            'urticaria': 'inflammatory', 'insect_bite': 'inflammatory'}

# 한 case 에 최대 3장이 같은 병변이므로 case_id 를 그룹 키로 쓴다.
scin = pd.read_csv(f'{SCIN_DIR}/metadata.csv')
scin_df = pd.DataFrame({
    'path'      : scin['img_id'].apply(lambda x: f'{SCIN_DIR}/images/{x}'),
    'label'     : scin['diagnostic'].map(SCIN2CLS),
    'scin_label': scin['diagnostic'],
    'group'     : 'scin_' + scin['case_id'].astype(str),
    'src'       : 'scin',
})

df = pd.concat([ham_df, pad_df, scin_df], ignore_index=True)
df['label_idx'] = df['label'].map(CLS2IDX)
assert df['label_idx'].notna().all(), '매핑 안 된 라벨이 있습니다'

print(df.groupby(['src', 'label']).size().unstack(fill_value=0))
print('총', len(df), '장 /', df['group'].nunique(), '병변')
print()
print('SCIN 원본 라벨 분포 (참고):')
print(scin_df['scin_label'].value_counts().to_string())

In [ ]:
# 6. 학습/검증 분할 -- 저장소에 고정된 홀드아웃 목록을 쓴다
#
# 예전에는 여기서 매번 StratifiedGroupKFold 를 새로 돌렸다. 문제는 그 결과가
# 어디에도 남지 않는다는 것이었다. 학습이 끝나면 "이 모델이 어떤 사진을 안 봤는지"를
# 아무도 말할 수 없고, 그러면 나중에 내는 어떤 평가 숫자도 근거가 없다.
# (실제로 그 일이 일어났다. 배포된 모델의 분할을 사후에 복원해야 했고,
#  복원이 맞는지는 끝내 증명하지 못했다.)
#
# 그래서 규칙을 둘로 못박는다.
#   1. 홀드아웃 목록의 사진은 절대 학습에 넣지 않는다.
#   2. 목록은 저장소(fastapi/tests/baselines/holdout.csv)에 커밋해 둔다.
# 이래야 모델을 몇 번을 갈아끼워도 같은 잣대로 비교할 수 있다.
import os
from sklearn.model_selection import StratifiedGroupKFold

HOLDOUT_CSV = f'{DRIVE}/holdout.csv'
SEED = 42

# 목록이 없을 때 새로 만드는 것을 기본으로 두지 않는다.
#
# 올리는 걸 깜빡했을 뿐인데 새 분할이 조용히 생기면, 학습은 겉보기에 멀쩡하게 끝나고
# 그 모델의 점수만 옛 모델과 비교할 수 없게 된다. 몇 시간을 태우고 나서야, 그것도
# 운이 좋아야 알아챈다. 그래서 여기서 멈추게 한다 -- 5분 늦는 편이 낫다.
#
# 데이터셋을 새로 구성해 **목록 자체를 다시 정할 때만** True 로 바꾼다.
ALLOW_NEW_HOLDOUT = False

# 목록은 파일명으로 맞춘다 -- 폴더 경로는 실행 환경마다 달라지기 때문이다.
df['image_id'] = df['path'].apply(os.path.basename)
assert df['image_id'].is_unique, '파일명이 겹칩니다 -- 목록으로 분할을 못 맞춥니다'

if os.path.exists(HOLDOUT_CSV):
    holdout_ids = set(pd.read_csv(HOLDOUT_CSV)['image_id'])
    in_holdout  = df['image_id'].isin(holdout_ids)
    missing = len(holdout_ids) - int(in_holdout.sum())
    assert missing == 0, (
        f'홀드아웃 목록의 {missing}장을 현재 데이터에서 찾지 못했습니다. '
        '데이터 구성이 목록과 다릅니다 -- 목록을 고치거나 데이터를 맞추세요. '
        '목록에 없는 사진으로 평가하면 지난 모델과 비교가 되지 않습니다.')
    valid_df, train_df = df[in_holdout], df[~in_holdout]
    print(f'홀드아웃 목록 사용: {HOLDOUT_CSV} ({len(holdout_ids)}장)')

elif not ALLOW_NEW_HOLDOUT:
    raise FileNotFoundError(
        f"""
홀드아웃 목록이 없습니다: {HOLDOUT_CSV}

이대로 두면 새 분할이 만들어지고, 이 모델의 점수는 지금까지의 기준값과
비교할 수 없게 됩니다. 학습을 시작하기 전에 목록을 올려 주세요.

  1. 저장소에서 파일을 받는다
       fastapi/tests/baselines/holdout.csv   (약 79KB)
  2. Drive 의 이 폴더에 그대로 올린다 (이름 바꾸지 말 것)
       MyDrive/artifact_medical_ai/holdout.csv
  3. 이 셀을 다시 실행한다

데이터셋을 새로 구성해서 목록 자체를 다시 정하려는 것이라면,
위의 ALLOW_NEW_HOLDOUT 를 True 로 바꾸고 실행하세요.
""")

else:
    # 목록을 새로 만드는 경로. 병변 단위로 나눈다 -- 사진 단위로 나누면 같은 병변을
    # 다른 각도에서 찍은 사진이 학습과 검증에 동시에 들어가, 점수가 실제보다 높게 나온다.
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, va_idx = next(sgkf.split(df, df['label_idx'], groups=df['group']))
    train_df, valid_df = df.iloc[tr_idx], df.iloc[va_idx]
    valid_df[['image_id', 'label', 'src']].to_csv(HOLDOUT_CSV, index=False)
    print(f'홀드아웃 목록을 새로 만들었습니다: {HOLDOUT_CSV}')
    print('  ⚠️ 이 파일을 fastapi/tests/baselines/holdout.csv 로 커밋하세요.')
    print('     커밋하지 않으면 다음 학습에서 또 다른 분할이 나오고, 모델 간 비교가 깨집니다.')
    print('  ⚠️ baselines/ 의 기존 기준 결과들은 이제 비교 대상이 아닙니다.')

# 누수 검사 -- 같은 병변이 양쪽에 있으면 홀드아웃은 '처음 보는 병변'이 아니다.
leak = set(train_df['group']) & set(valid_df['group'])
assert not leak, f'데이터 누수 -- 같은 병변이 학습과 홀드아웃에 동시에 있습니다: {len(leak)}개'

print(f'train {len(train_df)} / valid {len(valid_df)}')
print(valid_df['label'].value_counts())


In [ ]:
# 7. 클래스 가중치 (mel 이 2%대라 그냥 학습하면 mel 을 거의 못 맞춘다)
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# 가중치에 상한을 둔다.
# 직전 실험에서 df 는 표본이 100여 장뿐이라 가중치가 10.71 까지 올라갔고,
# 손실이 그 몇 장에 끌려다니면서 df 재현율이 0.963 -> 0.667 로 오히려 무너졌다.
# 희소 클래스를 살리되 학습이 그쪽으로 휘둘리지는 않게 한다.
W_CAP = 5.0

counts = train_df['label_idx'].value_counts().reindex(range(len(CLASSES))).fillna(0)
w = (counts.sum() / (len(CLASSES) * counts.clip(lower=1))).clip(upper=W_CAP)
weights = torch.tensor(w.values, dtype=torch.float, device=device)

# label_smoothing: SCIN 라벨은 의사 3인의 가중 투표라 정답 자체가 100% 확신이 아니다.
# 모델이 한 클래스에 확률 1.0 을 몰아주지 않게 살짝 눌러 준다(과적합도 함께 줄어든다).
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

print(dict(zip(CLASSES, w.round(2).values)))
print('학습 표본:', dict(zip(CLASSES, counts.astype(int).values)))

In [ ]:
# 8. Dataset / DataLoader
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

IMAGE_SIZE, BATCH_SIZE = 224, 32

# 직전 실험은 3 epoch 째부터 검증 손실이 계속 올라갔다(과적합).
# 크롭과 색상을 흔들어 같은 사진을 매번 조금씩 다르게 보여준다.
# 피부 병변 사진에는 정해진 위아래가 없으므로 수직 뒤집기도 안전하다.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinLesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        image = self.transform(image)
        return image, int(row['label_idx'])

train_loader = DataLoader(SkinLesionDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(SkinLesionDataset(valid_df, valid_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'학습 배치 수: {len(train_loader)}')

In [ ]:
# 9. 모델 준비
# 클래스 수가 7 -> 8 로 늘어, 기존 model.pth 를 그대로 load_state_dict 하면
# 분류 헤드(classifier)의 크기가 안 맞아 터진다.
# 그래서 **백본 가중치만 물려받고 헤드는 새로 초기화**한다 —
# 피부 사진에서 특징을 뽑는 능력은 그대로 가져오고, 병명을 고르는 마지막 층만 다시 배운다.
import timm
import os

EXISTING = f'{DRIVE}/model.pth'

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(CLASSES))

if os.path.exists(EXISTING):
    old = torch.load(EXISTING, map_location='cpu', weights_only=True)
    cur = model.state_dict()
    # 이름이 같고 모양도 같은 것만 가져온다(= classifier.weight/bias 만 자동으로 빠진다)
    usable = {k: v for k, v in old.items()
              if k in cur and cur[k].shape == v.shape}
    skipped = [k for k in old if k not in usable]
    model.load_state_dict(usable, strict=False)
    print(f'기존 모델에서 {len(usable)}개 텐서 물려받음: {EXISTING}')
    print(f'  새로 초기화된 층: {skipped}')
else:
    print('model.pth 를 못 찾아 ImageNet 사전학습 가중치로 시작')

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'출력 클래스 수: {len(CLASSES)}')

In [ ]:
# 10. 학습 루프
from sklearn.metrics import recall_score

def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
        preds += outputs.argmax(1).cpu().tolist()
        labels_all += labels.cpu().tolist()
    return total_loss / len(loader), preds, labels_all

EPOCHS = 15
MEL_IDX = CLASSES.index('mel')

# 학습률을 코사인으로 서서히 낮춘다.
# 직전 실험은 lr 이 끝까지 1e-4 로 고정이라 epoch 마다 결과가 크게 출렁였고,
# 그 출렁임의 꼭대기가 '최고 모델'로 뽑히는 사고가 났다(mel 0.68 은 앞뒤가 0.53 인 스파이크였다).
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 체크포인트 기준: mel 재현율만 보면 한 epoch 튄 값에 속는다.
# mel 을 여전히 가장 무겁게 두되, 8개 클래스 평균 재현율을 섞어 흔들림을 눌러 준다.
def selection_score(recalls):
    return 0.6 * recalls[MEL_IDX] + 0.4 * recalls.mean()

best_score = -1
best_epoch = -1
history = {'train_loss': [], 'val_loss': [], 'mel_recall': [], 'macro_recall': []}

for epoch in range(EPOCHS):
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_preds, val_labels = run_epoch(valid_loader, False)
    scheduler.step()

    recalls = recall_score(val_labels, val_preds, average=None,
                           labels=range(len(CLASSES)), zero_division=0)
    mel_recall   = recalls[MEL_IDX]
    macro_recall = recalls.mean()
    score        = selection_score(recalls)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['mel_recall'].append(mel_recall)
    history['macro_recall'].append(macro_recall)

    mark = ''
    if score > best_score:
        best_score, best_epoch = score, epoch + 1
        torch.save(model.state_dict(), f'{DRIVE}/model_v4_best.pth')
        mark = '  <- 저장'

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | '
          f'mel {mel_recall:.3f} | macro {macro_recall:.3f} | score {score:.3f}{mark}')

import numpy as np
mel_hist = np.array(history['mel_recall'])
print(f'\n학습 완료. 저장된 epoch: {best_epoch}  (mel {mel_hist[best_epoch-1]:.3f})')
print(f'mel 재현율 전체 평균: {mel_hist.mean():.3f}  /  최고: {mel_hist.max():.3f}  '
      f'/  뒤쪽 5 epoch 평균: {mel_hist[-5:].mean():.3f}')
print('※ 저장된 값이 전체 평균보다 크게 높으면 스파이크를 집은 것이니 의심해야 한다.')

In [ ]:
# 11. 학습 곡선
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='학습 손실')
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='검증 손실')
ax1.set_title('손실 (Loss)')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history['mel_recall'], 'g-o', label='mel 재현율')
ax2.plot(epochs_range, history['macro_recall'], 'c-o', label='전체 평균 재현율')
# 실제로 저장된 지점을 표시한다. 이 점이 곡선에서 혼자 튀어 있으면
# 우연히 잘 나온 epoch 을 집은 것이므로 그 체크포인트는 믿으면 안 된다.
ax2.axvline(best_epoch, color='k', ls='--', lw=1, label=f'저장된 epoch ({best_epoch})')
ax2.set_title('흑색종(mel) 재현율')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True)

plt.suptitle(f'학습 결과 (저장 시점 mel 재현율: {history["mel_recall"][best_epoch-1]:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# 12. 최종 평가 -- 클래스별 재현율 / 혼동 행렬
from sklearn.metrics import classification_report, confusion_matrix, precision_score
import seaborn as sns

model.load_state_dict(torch.load(f'{DRIVE}/model_v4_best.pth',
                                 map_location=device, weights_only=True))
model.eval()
_, val_preds, val_labels = run_epoch(valid_loader, False)

print(classification_report(val_labels, val_preds, target_names=CLASSES,
                            labels=range(len(CLASSES)), digits=3, zero_division=0))

cm = confusion_matrix(val_labels, val_preds, labels=range(len(CLASSES)))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('혼동 행렬 (8 클래스)')
plt.ylabel('실제 병명')
plt.xlabel('예측 병명')
plt.tight_layout()
plt.show()

# ── 합격 판정 ──────────────────────────────────────────────────
# 기존 배포 모델(v2, 7클래스)의 검증 성적. 이 아래로 떨어지면 배포하지 않는다.
BASELINE_MEL = 0.729

r = recall_score(val_labels, val_preds, average=None,
                 labels=range(len(CLASSES)), zero_division=0)
p = precision_score(val_labels, val_preds, average=None,
                    labels=range(len(CLASSES)), zero_division=0)
INF_IDX = CLASSES.index('inflammatory')

mel_r, inf_r, inf_p = r[MEL_IDX], r[INF_IDX], p[INF_IDX]
ok_mel = mel_r >= BASELINE_MEL - 0.03
ok_inf = inf_r >= 0.90 and inf_p >= 0.85

print('=' * 52)
print(f"mel 재현율        {mel_r:.3f}   (기준 {BASELINE_MEL - 0.03:.3f} 이상) "
      f"{'통과' if ok_mel else '실패'}")
print(f"inflammatory      재현율 {inf_r:.3f} / 정밀도 {inf_p:.3f}   "
      f"(기준 0.90 / 0.85) {'통과' if ok_inf else '실패'}")
print('=' * 52)
print('→ 배포 가능' if (ok_mel and ok_inf) else '→ 배포 불가. 3단계로 넘어가지 말 것.')

In [ ]:
# 13. PAD 데이터만 따로 평가 (실제 키오스크 촬영 환경에 가장 가까운 지표)
pad_valid = valid_df[valid_df['src'] == 'pad']
if len(pad_valid) > 0:
    pad_loader = DataLoader(SkinLesionDataset(pad_valid, valid_transform),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    _, pad_preds, pad_labels = run_epoch(pad_loader, False)
    print(f'PAD 검증 {len(pad_valid)}장')
    # PAD 에는 df/vasc 표본이 아예 없어 labels 를 명시하지 않으면
    # classification_report 가 target_names 길이와 안 맞는다며 에러를 낸다.
    print(classification_report(pad_labels, pad_preds, target_names=CLASSES,
                                labels=range(len(CLASSES)), digits=3, zero_division=0))
else:
    print('이번 분할에서 valid 에 PAD 표본이 없습니다. 셀 6을 다시 실행해보세요.')

In [ ]:
# 14. SCIN 데이터만 따로 평가 (통합한 염증성 클래스가 실제로 학습됐는지)
scin_valid = valid_df[valid_df['src'] == 'scin']
if len(scin_valid) > 0:
    scin_loader = DataLoader(SkinLesionDataset(scin_valid, valid_transform),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    _, scin_preds, scin_labels = run_epoch(scin_loader, False)
    print(f'SCIN 검증 {len(scin_valid)}장')
    print(classification_report(scin_labels, scin_preds, target_names=CLASSES,
                                labels=range(len(CLASSES)), digits=3, zero_division=0))

    # SCIN 사진 중 색소성 7클래스로 잘못 새어나간 비율.
    # 키오스크에서 습진 사진에 '흑색종' 이 뜨는 사고가 이 숫자다.
    leaked = sum(1 for q in scin_preds if q != CLASSES.index('inflammatory'))
    print(f'색소성으로 잘못 분류된 SCIN 사진: {leaked} / {len(scin_preds)} '
          f'({leaked / len(scin_preds) * 100:.1f}%)')

    # 원본 4개 라벨별로 나눠 봤을 때 어느 쪽이 특히 새는지
    sv = scin_valid.reset_index(drop=True)
    sv['pred_ok'] = [q == CLASSES.index('inflammatory') for q in scin_preds]
    print()
    print('원본 라벨별 염증성 인식률:')
    print((sv.groupby('scin_label')['pred_ok'].agg(['mean', 'size'])
             .rename(columns={'mean': '인식률', 'size': '장수'})
             .round(3).to_string()))
else:
    print('이번 분할에서 valid 에 SCIN 표본이 없습니다. 셀 6을 다시 실행해보세요.')

## 다음 단계

셀 12의 합격 판정이 **배포 가능**으로 나왔을 때만 진행합니다.

1. Drive 의 `model_v4_best.pth` 를 내려받아 `fastapi/model.pth` 로 교체
2. `fastapi/main.py` 의 `CLASSES` 를 위 셀 5 와 **똑같은 순서**로 8개로 바꾸고,
   `CLASS_NAMES_KO` 에 `inflammatory` 추가, `num_classes=8` 로 수정
3. 백엔드/프론트/DB 반영 — 계획서 3단계 참고
   (클래스가 8개라 `Map.of` 10쌍 한도에는 걸리지 않는다)
4. `docker compose build fastapi backend && docker compose up -d fastapi backend`

모델 버전은 가중치 파일 해시로 자동 계산되므로 따로 손댈 것이 없다.

### 합격 기준
- **mel 재현율 ≥ 0.699** (기존 배포 모델 0.729 대비 3포인트 이내).
  새 클래스를 넣다가 흑색종을 놓치기 시작하면 임상적으로 최악이다. 이게 1순위다.
- **inflammatory 재현율 ≥ 0.90, 정밀도 ≥ 0.85.**
  "염증성 피부질환이고 피부암 계열은 아니다"라고 말할 수 있어야 의미가 있다.
  직전 11클래스 실험에서 이 통합 기준으로는 재현율 0.987 / 정밀도 0.968 이 나왔다.
- 저장된 epoch 의 mel 재현율이 전체 평균과 크게 벌어지지 않을 것(스파이크 방지).